<a href="https://colab.research.google.com/github/MonnikLuianne/wildfire-risk-piaui/blob/main/wildfire_risk_piaui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Wildfire Risk Prediction in Piauí
   **Author:** Monnik Luianne Oliveira Nascimento Veloso Barbosa

## 1. Setup and Reproducibility
Libraries, versions and random seed used in this notebook.

In [1]:
import sys, random
import numpy as np
import pandas as pd
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.18
pandas: 2.2.3
numpy: 2.4.6
scikit-learn: 1.6.1


## 2. Data Loading
Active fire detections (INPE – BDQueimadas, reference satellite AQUA_M-T), Piauí, 2015–2025.

In [2]:
BASE_URL = "https://raw.githubusercontent.com/MonnikLuianne/wildfire-risk-piaui/main/data/raw"
YEARS = range(2015, 2026)

frames = []
for year in YEARS:
    df_year = pd.read_csv(f"{BASE_URL}/focos_PI_{year}.csv")
    frames.append(df_year)
    print(year, len(df_year))

fires = pd.concat(frames, ignore_index=True)
print("Total:", fires.shape)
fires.head()

2015 13839
2016 8303
2017 7657
2018 9431
2019 10894
2020 9317
2021 12745
2022 10866
2023 12957
2024 10587
2025 11806
Total: (118402, 12)


,DataHora,Satelite,Pais,Estado,Municipio,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,FRP,Latitude,Longitude
0,2015/01/01 17:05:00,AQUA_M-T,Brasil,PIAUÍ,CANAVIEIRA,Cerrado,0.0,0.3,0.0,28.3,-7.268,-43.717
1,2015/01/01 17:06:00,AQUA_M-T,Brasil,PIAUÍ,PIRACURUCA,Caatinga,1.0,0.0,1.0,80.5,-3.859,-41.591
2,2015/01/01 17:06:00,AQUA_M-T,Brasil,PIAUÍ,PIRACURUCA,Caatinga,1.0,0.0,1.0,101.9,-3.861,-41.595
3,2015/01/02 16:09:00,AQUA_M-T,Brasil,PIAUÍ,BOM JESUS,Cerrado,2.0,0.0,1.0,48.1,-9.149,-44.568
4,2015/01/02 16:09:00,AQUA_M-T,Brasil,PIAUÍ,MONTE ALEGRE DO PIAUÍ,Cerrado,2.0,0.0,1.0,27.8,-9.469,-44.831


### 2.1 Municipalities of Piauí
Coordinates (municipal seat) from the IBGE-based dataset by Kelvin S. do Prado (github.com/kelvins/municipios-brasileiros).

In [3]:
import unicodedata

def normalize(name):
    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return name.upper().strip()

MUN_URL = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"
municipalities = pd.read_csv(MUN_URL)
municipalities = municipalities[municipalities["codigo_uf"] == 22][["codigo_ibge", "nome", "latitude", "longitude"]]
municipalities["mun_key"] = municipalities["nome"].map(normalize)
print("Municipalities in Piauí:", len(municipalities))

fires["mun_key"] = fires["Municipio"].map(normalize)
unmatched = set(fires["mun_key"]) - set(municipalities["mun_key"])
print("Unmatched names:", unmatched)

Municipalities in Piauí: 224
Unmatched names: set()


### 2.2 Weather data (NASA POWER)
Daily weather for each municipality, obtained from the NASA POWER API (power.larc.nasa.gov), 2015–2025.
Variables: max temperature (T2M_MAX), mean temperature (T2M), relative humidity (RH2M), precipitation (PRECTOTCORR) and wind speed (WS2M).

In [4]:
import requests

POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMS = ["T2M_MAX", "T2M", "RH2M", "PRECTOTCORR", "WS2M"]

def get_weather(lat, lon, start="20150101", end="20251231"):
    query = {
        "parameters": ",".join(PARAMS),
        "community": "AG",
        "latitude": lat,
        "longitude": lon,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    r = requests.get(POWER_URL, params=query, timeout=120)
    r.raise_for_status()
    data = r.json()["properties"]["parameter"]
    df = pd.DataFrame(data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "date"
    return df.replace(-999, np.nan)

# Test with Teresina
the = municipalities[municipalities["mun_key"] == "TERESINA"].iloc[0]
test = get_weather(the["latitude"], the["longitude"])
print(test.shape)
print(test.isna().sum())
test.head()

(4018, 5)
T2M_MAX        0
T2M            0
RH2M           0
PRECTOTCORR    0
WS2M           0
dtype: int64


,T2M_MAX,T2M,RH2M,PRECTOTCORR,WS2M
date,,,,,
2015-01-01,31.87,27.59,66.72,0.93,1.08
2015-01-02,34.55,28.81,63.03,4.44,0.96
2015-01-03,26.37,24.36,85.66,8.81,0.92
2015-01-04,32.73,26.63,74.21,1.98,1.08
2015-01-05,32.43,27.06,71.00,4.57,1.15


In [5]:
import time
from tqdm.auto import tqdm

results = {}

for _, row in tqdm(municipalities.iterrows(), total=len(municipalities)):
    key = row["mun_key"]
    if key in results:
        continue  # already downloaded
    for attempt in range(3):
        try:
            w = get_weather(row["latitude"], row["longitude"])
            w["mun_key"] = key
            results[key] = w
            break
        except Exception as e:
            print(key, "- attempt", attempt + 1, "failed:", e)
            time.sleep(10)
    time.sleep(1)  # be gentle with the API

weather = pd.concat(results.values()).reset_index()
print(weather.shape, "| municipalities:", weather["mun_key"].nunique())
print(weather.isna().sum())

weather.to_csv("weather_PI_2015_2025.csv.gz", index=False, compression="gzip")

  0%|          | 0/224 [00:00<?, ?it/s]

(900032, 7) | municipalities: 224
date           0
T2M_MAX        0
T2M            0
RH2M           0
PRECTOTCORR    0
WS2M           0
mun_key        0
dtype: int64
